# DTL: classify probability map

In [1]:
# CONFIG

VERSION = 'v7'

# 1. Load probability stack asset from step 3

In [2]:
!python -m pip install .. --quiet

import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

# This notebook is to test one province only

province_name = 'Aceh'

# Load probability stack from asset

prob_stack = ee.Image(f'projects/epistem2/assets/probability_stack_{province_name}_2021_{VERSION}')

provinces = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra_Provinces')
province_list = provinces.toList(provinces.size())
# aoi = ee.Feature(province_list.get(0)).geometry()


aoi = (provinces
    .filter(ee.Filter.eq('AoI', province_name))
    .first()
    .geometry()
)


stacked_landsat = ee.Image(f'projects/epistem2/assets/stacked_landsat_2021_sumatra_{VERSION}')
band_names = stacked_landsat.bandNames()


Successfully saved authorization token.


# 2. Classify the prob maps

Using `Argmax`

In [3]:
# Derive class_ids directly from prob_stack's band names — guaranteed to match
prob_band_names = prob_stack.bandNames().getInfo()  # e.g. ['prob_18', 'prob_2', ...]

# Extract the numeric class id from each band name, keep prob_stack's own order
class_ids_list = [int(b.split('_')[1]) for b in prob_band_names]
class_ids = ee.List(class_ids_list)

print("class_ids (from prob_stack):", class_ids_list)

# prob_stack is already in this exact order — no need to reorder/select
max_prob_index = prob_stack.toArray().arrayArgmax().arrayGet(0)

final_lc = max_prob_index.remap(
    ee.List.sequence(0, class_ids.size().subtract(1)),
    class_ids
).rename('classification')

max_confidence = prob_stack.toArray().arrayReduce(ee.Reducer.max(), [0]).arrayGet([0]).rename('confidence')

class_ids (from prob_stack): [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24]


## Optional: visualization and statistic check

In [4]:
import geemap

Map = geemap.Map()
Map.centerObject(aoi, 9)

class_ids_list = class_ids.getInfo()
class_palette = ['e31a1c','33a02c','1f78b4','ff7f00','b15928',
                  '6a3d9a','a6cee3','b2df8a','fb9a99','fdbf6f','cab2d6']

n_classes = class_ids.size().getInfo()

classification_vis = {
    'min': min(class_ids_list),
    'max': max(class_ids_list),
    'palette': class_palette[:len(class_ids_list)]
}


confidence_vis = {
    'min': 0,
    'max': 1,
    'palette': ['ffffcc', 'ffeda0', 'fed976', 'feb24c', 'fd8d3c', 'fc4e2a', 'e31a1c', 'b10026']
}

Map.addLayer(aoi, {}, 'AOI', True)
Map.addLayer(final_lc.clip(aoi), classification_vis, 'Final Classification')
Map.addLayer(max_confidence.clip(aoi), confidence_vis, 'Confidence', False)

# Map.add_legend(
#     title="Land Cover Class",
#     labels=[str(c) for c in class_ids.getInfo()],
#     colors=class_palette[:n_classes]
# )

Map

Map(center=[4.225359204755897, 96.9124222729065], controls=(WidgetControl(options=['position', 'transparent_bg…

In [5]:
# %%
# Compute area per class using pixelArea + group reducer
area_image = ee.Image.pixelArea().addBands(final_lc)

area_stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName='class'
    ),
    geometry=aoi,
    scale=100,
    maxPixels=1e13,
    bestEffort=True
)

area_stats_info = area_stats.getInfo()
print(area_stats_info)

# %%
import pandas as pd

# Parse into a clean DataFrame
groups = area_stats_info['groups']

df_stats = pd.DataFrame(groups)
df_stats = df_stats.rename(columns={'class': 'class_id', 'sum': 'area_m2'})
df_stats['area_ha'] = df_stats['area_m2'] / 10_000
df_stats['area_km2'] = df_stats['area_m2'] / 1_000_000
df_stats['pct_of_total'] = (df_stats['area_m2'] / df_stats['area_m2'].sum()) * 100

df_stats = df_stats.sort_values('area_ha', ascending=False).reset_index(drop=True)
df_stats

# %%
# Pixel counts per class (useful sanity check alongside area)
pixel_count_stats = final_lc.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=100,
    maxPixels=1e13,
    bestEffort=True
).getInfo()

pixel_counts = pixel_count_stats['classification']
df_pixels = pd.DataFrame(
    [{'class_id': int(k), 'pixel_count': v} for k, v in pixel_counts.items()]
).sort_values('pixel_count', ascending=False).reset_index(drop=True)

df_pixels

# %%
# Merge for a single summary table
df_summary = df_stats.merge(df_pixels, on='class_id')
df_summary = df_summary[['class_id', 'pixel_count', 'area_ha', 'area_km2', 'pct_of_total']]
df_summary

{'groups': [{'class': 1, 'sum': 22340734009.973106}, {'class': 2, 'sum': 20549274838.615845}, {'class': 3, 'sum': 5448694.355269608}, {'class': 4, 'sum': 523892262.04702055}, {'class': 5, 'sum': 4108953.170021446}, {'class': 6, 'sum': 2842533229.4494634}, {'class': 9, 'sum': 6512410112.942655}, {'class': 16, 'sum': 158409.75}, {'class': 17, 'sum': 42934448.52036612}, {'class': 18, 'sum': 1384838257.2899396}, {'class': 20, 'sum': 29704.736328125}, {'class': 23, 'sum': 2473435807.316693}]}


,class_id,pixel_count,area_ha,area_km2,pct_of_total
0,1,2.255091e+06,2.234073e+06,22340.734010,39.415690
1,2,2.074439e+06,2.054927e+06,20549.274839,36.255024
2,9,6.574227e+05,6.512410e+05,6512.410113,11.489826
3,6,2.870487e+05,2.842533e+05,2842.533229,5.015073
4,23,2.495862e+05,2.473436e+05,2473.435807,4.363875
5,18,1.398133e+05,1.384838e+05,1384.838257,2.443266
6,4,5.286851e+04,5.238923e+04,523.892262,0.924302
7,17,4.334694e+03,4.293445e+03,42.934449,0.075749
8,3,5.502667e+02,5.448694e+02,5.448694,0.009613
9,5,4.146706e+02,4.108953e+02,4.108953,0.007249


In [ ]:
# quick bar chart of area by class
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df_summary['class_id'].astype(str), df_summary['area_ha'], color='steelblue')
ax.set_xlabel('Class ID')
ax.set_ylabel('Area (ha)')
ax.set_title('Land Cover Class Area — Final Classification')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [6]:
# confidence stats per class (mean confidence for pixels assigned to each class)
confidence_by_class = max_confidence.addBands(final_lc).reduceRegion(
    reducer=ee.Reducer.mean().group(
        groupField=1,
        groupName='class'
    ),
    geometry=aoi,
    scale=100,
    maxPixels=1e13,
    bestEffort=True
).getInfo()

df_conf = pd.DataFrame(confidence_by_class['groups']).rename(
    columns={'class': 'class_id', 'mean': 'mean_confidence'}
)

df_summary_full = df_summary.merge(df_conf, on='class_id')
df_summary_full

,class_id,pixel_count,area_ha,area_km2,pct_of_total,mean_confidence
0,1,2.255091e+06,2.234073e+06,22340.734010,39.415690,76.784446
1,2,2.074439e+06,2.054927e+06,20549.274839,36.255024,74.170382
2,9,6.574227e+05,6.512410e+05,6512.410113,11.489826,69.038802
3,6,2.870487e+05,2.842533e+05,2842.533229,5.015073,63.292336
4,23,2.495862e+05,2.473436e+05,2473.435807,4.363875,76.258893
5,18,1.398133e+05,1.384838e+05,1384.838257,2.443266,73.656331
6,4,5.286851e+04,5.238923e+04,523.892262,0.924302,69.634267
7,17,4.334694e+03,4.293445e+03,42.934449,0.075749,65.655078
8,3,5.502667e+02,5.448694e+02,5.448694,0.009613,50.000000
9,5,4.146706e+02,4.108953e+02,4.108953,0.007249,50.954210


# Post-classification refinement

In [7]:
# load luma-ge predictors

from luma_ge.predictor import terrain_calculator, SpectralCalculator, distance_calculator
dist_calc = distance_calculator()
terrain_calc = terrain_calculator()
spectral_calc = SpectralCalculator()
#Define DEM
#retrieve topography based predictors
dem_source = 'NASADEM'
elevation = terrain_calc.calculate_elevation(aoi, dem_source=dem_source) # type: ignore
slope = terrain_calc.calculate_slope(aoi, dem_source=dem_source) # type: ignore
# aspect = terrain_calc.calculate_aspect(aoi, dem_source=dem_source) # type: ignore

#distance based predictors (5km buffer)
distance_predictor = dist_calc.calculate_distance_metrics(
    aoi = aoi, max_dist = 50000, in_meters = True)

2026-09-15 10:02:40,984 - INFO - Earth Engine initialized successfully
2026-09-15 10:02:40,986 - INFO - Distance calculator initialized
2026-09-15 10:02:40,988 - INFO - Terrain calculator initialized
2026-09-15 10:02:47,506 - INFO - SpectralCalculator initialized
2026-09-15 10:02:47,508 - INFO - Calculating elevation layer using NASADEM DEM...
2026-09-15 10:02:47,509 - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-09-15 10:02:47,510 - INFO - Calculating slope layer using NASADEM DEM...
2026-09-15 10:02:47,512 - INFO - Calculating elevation layer using NASADEM DEM...
2026-09-15 10:02:47,514 - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-09-15 10:02:47,515 - INFO - Successfully calculated slope layer using NASADEM DEM
2026-09-15 10:02:47,517 - INFO - Calculating distance metrics (max_dist=50000, in_meters=True)...
2026-09-15 10:02:47,518 - INFO - Using meter-based distance calculation (slower but more accurate)
2026-09-15 10:02:47,519 - IN

In [8]:
predictor_stack  = elevation.addBands(slope).addBands(distance_predictor).toFloat()
print('Final predictor stack bands:', predictor_stack.bandNames().getInfo() if predictor_stack else 'failed')

Final predictor stack bands: ['elevation', 'slope', 'dist_roads', 'dist_coast', 'dist_settlement']


## Apply ruleset

Modified function from Faza to run fully on server side

In [9]:
def build_validity_stack_ee(predictor_image, class_ids, ruleset_df, predictor_map=None):
    """
    Build a boolean validity mask per class, as a single multi-band ee.Image.

    Parameters
    ----------
    predictor_image : ee.Image
        Multi-band image containing the predictor bands referenced by
        predictor_map (e.g. 'elevation', distance bands, ...).
    class_ids : list
        Class_ID values, in the same order as prob_stack's bands.
    ruleset_df : pandas.DataFrame
        Indexed by Class_ID, with '<prefix>_min' / '<prefix>_max' columns.
    predictor_map : dict, optional
        Maps predictor band name -> rule column prefix. Defaults to
        DEFAULT_PREDICTOR_MAP.

    Returns
    -------
    ee.Image
        One band per class, named 'valid_<class_id>', in the input order.
        1 = pixel passes the rules for that class, 0 = it doesn't.
    """
    if predictor_map is None:
        predictor_map = DEFAULT_PREDICTOR_MAP

    predictor_band_names = predictor_image.bandNames().getInfo()

    valid_bands = []
    missing_classes = []

    for cid in class_ids:
        if cid not in ruleset_df.index:
            missing_classes.append(cid)
            valid_bands.append(ee.Image.constant(1).toByte().rename(f"valid_{cid}"))
            continue

        rule = ruleset_df.loc[cid]
        class_valid = ee.Image.constant(1).toByte()

        for band_name, prefix in predictor_map.items():
            if band_name not in predictor_band_names:
                continue
            min_col, max_col = f"{prefix}_min", f"{prefix}_max"
            if min_col not in rule or max_col not in rule:
                continue

            lo, hi = rule[min_col], rule[max_col]
            band = predictor_image.select(band_name)

            ok = ee.Image.constant(1).toByte()
            if pd.notna(lo):
                ok = ok.And(band.gte(float(lo)))
            if pd.notna(hi):
                ok = ok.And(band.lte(float(hi)))

            # masked / nodata predictor pixels are treated as valid,
            # mirroring the `ok |= np.isnan(vals)` behaviour of the original
            ok = ok.Or(band.mask().Not())

            class_valid = class_valid.And(ok)

        valid_bands.append(class_valid.rename(f"valid_{cid}"))

    if missing_classes:
        print(f"Warning: no ruleset row for IDs {sorted(set(missing_classes))}; unconstrained.")

    return ee.Image.cat(valid_bands)

def predict_class_ruleset_ee(valid_image, prob_stack, final_lc, max_confidence,
                              class_ids, aoi=None, scale=100, report_stats=True):
    """
    Argmax classification followed by rule-based post-processing, entirely
    server-side.

    Parameters
    ----------
    valid_image : ee.Image
        Output of build_validity_stack_ee — one 'valid_<id>' band per class,
        in the same order as prob_stack's bands.
    prob_stack : ee.Image
        Raw per-class probability image ('prob_<id>' bands), prob_stack from
        cell 3/5.
    final_lc : ee.Image
        Raw argmax classification (from cell 5), used as the fallback where
        no class is valid.
    max_confidence : ee.Image
        Raw argmax probability (from cell 5).
    class_ids : list
        Class_ID values in prob_stack band order (class_ids_list).
    aoi : ee.Geometry, optional
        Region to compute the "pixels changed" / "no valid class" counts
        over. Required only if report_stats=True.
    scale : int
        Scale (m) for the stats reduceRegion calls.
    report_stats : bool
        If True, triggers a couple of reduceRegion calls (like the original
        print statements) to report how many pixels changed / had no valid
        class. Set False to keep everything lazy/server-side.

    Returns
    -------
    dict with keys: classified_map, max_prob_map, raw_classified_map,
                     validity_stack, corrected_mask, no_valid_class_mask.
        All values are ee.Image objects (single band each, except
        validity_stack which keeps one band per class).
    """
    class_ids_list = list(class_ids)
    n = len(class_ids_list)

    # Large penalty pushes invalid classes below any real probability,
    # without disturbing relative ordering among valid classes.
    PENALTY = 1e6
    penalty = valid_image.subtract(1).multiply(PENALTY)  # 0 if valid=1, -PENALTY if valid=0
    penalty = penalty.rename(prob_stack.bandNames())      # align band-for-band with prob_stack

    adjusted_probs = prob_stack.add(penalty)
    adjusted_array = adjusted_probs.toArray()

    adjusted_argmax = adjusted_array.arrayArgmax().arrayGet(0)
    adjusted_maxprob = adjusted_array.arrayReduce(ee.Reducer.max(), [0]).arrayGet([0])

    adjusted_lc = adjusted_argmax.remap(
        ee.List.sequence(0, n - 1),
        ee.List(class_ids_list)
    ).rename("classification")

    # true where every class failed its rules for that pixel
    no_valid = valid_image.reduce(ee.Reducer.max()).eq(0).rename("no_valid_class")

    # fall back to the raw argmax/confidence where nothing passed the rules
    final_lc_corrected = final_lc.where(no_valid.eq(0), adjusted_lc).rename("classification")
    final_confidence = max_confidence.where(no_valid.eq(0), adjusted_maxprob).rename("confidence")

    # mask out pixels the raw classifier already had zero confidence for
    final_lc_corrected = final_lc_corrected.where(max_confidence.eq(0), 0)

    corrected = final_lc_corrected.neq(final_lc).And(max_confidence.neq(0)).rename("corrected")

    if report_stats:
        if aoi is None:
            print("report_stats=True but no aoi given; skipping pixel-count summary.")
        else:
            corrected_count = corrected.reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=scale,
                maxPixels=1e13, bestEffort=True
            ).get("corrected").getInfo()
            no_valid_count = no_valid.reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=scale,
                maxPixels=1e13, bestEffort=True
            ).get("no_valid_class").getInfo()
            print(f"Note: {int(no_valid_count or 0)} pixel(s) with no valid class; fallback to raw argmax.")
            print(f"Ruleset changed {int(corrected_count or 0)} pixels.")

    return {
        "classified_map": final_lc_corrected,
        "max_prob_map": final_confidence,
        "raw_classified_map": final_lc,
        "validity_stack": valid_image,
        "corrected_mask": corrected,
        "no_valid_class_mask": no_valid,
    }

In [10]:
# BAND NAME MAPPING

DEFAULT_PREDICTOR_MAP = {
    "elevation": "dem",
    "slope": "slope",
    "dist_coast": "dist_coast",
    # "aceh_worldpop_2020_30m_ne": "worldpop",
}

ruleset_df = pd.read_excel('../data/modular_mapping_approach/postclassification_predictors_ruleset.xlsx', na_values=['-', ''])
ruleset_df = ruleset_df.set_index("Class_ID", drop=False)  

valid_image = build_validity_stack_ee(
    predictor_stack, class_ids_list, ruleset_df, DEFAULT_PREDICTOR_MAP
)

result = predict_class_ruleset_ee(
    valid_image, prob_stack, final_lc, max_confidence, class_ids_list,
    aoi=aoi, scale=100
)

final_lc_refined = result["classified_map"]
final_confidence_refined = result["max_prob_map"]

2026-09-15 10:03:31,775 - WARNING - Sleeping 0.57 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/397031820099/value:compute?prettyPrint=false&alt=json, after 503


Note: 36595 pixel(s) with no valid class; fallback to raw argmax.
Ruleset changed 375985 pixels.


In [11]:
raw_lc = result["raw_classified_map"]
corrected_lc = result["classified_map"]
corrected_mask = result["corrected_mask"]
no_valid_mask = result["no_valid_class_mask"]

# change-status image: 0=background, 1=unchanged, 2=corrected, 3=fallback (no valid class)
status = ee.Image(1).byte() \
    .where(raw_lc.eq(0), 0) \
    .where(corrected_mask, 2) \
    .where(no_valid_mask.And(raw_lc.neq(0)), 3) \
    .rename('status')

status_vis = {'min': 0, 'max': 3, 'palette': ['e0e0e0', 'd9f0d3', 'e31a1c', '6a3d9a']}

# --- Map 1: everything layered together ---
Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(raw_lc.clip(aoi), classification_vis, 'Raw Argmax')
Map.addLayer(corrected_lc.clip(aoi), classification_vis, 'Ruleset-Corrected')
Map.addLayer(status.clip(aoi), status_vis, 'Change Status')
# Map.add_legend(
#     title="Change status",
#     labels=["Background", "Unchanged", "Corrected", "Fallback"],
#     colors=["#e0e0e0", "#d9f0d3", "#e31a1c", "#6a3d9a"]
# )
Map

# --- Map 2: swipe comparison, raw vs corrected side by side ---

# swipe_map = geemap.Map()
# swipe_map.centerObject(aoi, 9)
# left_layer = geemap.ee_tile_layer(raw_lc.clip(aoi), classification_vis, 'Raw Argmax')
# right_layer = geemap.ee_tile_layer(corrected_lc.clip(aoi), classification_vis, 'Ruleset-Corrected')
# swipe_map.split_map(left_layer, right_layer)
# swipe_map

Map(center=[4.2253592047558985, 96.9124222729065], controls=(WidgetControl(options=['position', 'transparent_b…

In [12]:
# --- Stats: per-class area, raw vs corrected, computed server-side ---
raw_area = ee.Image.pixelArea().addBands(raw_lc).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
    geometry=aoi, scale=100, maxPixels=1e13, bestEffort=True
).getInfo()

corrected_area = ee.Image.pixelArea().addBands(corrected_lc).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
    geometry=aoi, scale=100, maxPixels=1e13, bestEffort=True
).getInfo()

df_raw = pd.DataFrame(raw_area['groups']).rename(columns={'class': 'Class_ID', 'sum': 'area_m2_raw'})
df_corr = pd.DataFrame(corrected_area['groups']).rename(columns={'class': 'Class_ID', 'sum': 'area_m2_corrected'})

df_compare = df_raw.merge(df_corr, on='Class_ID', how='outer').fillna(0)
df_compare['area_ha_raw'] = df_compare['area_m2_raw'] / 10_000
df_compare['area_ha_corrected'] = df_compare['area_m2_corrected'] / 10_000
df_compare['change_ha'] = df_compare['area_ha_corrected'] - df_compare['area_ha_raw']
df_compare = df_compare.sort_values('Class_ID').reset_index(drop=True)
df_compare[['Class_ID', 'area_ha_raw', 'area_ha_corrected', 'change_ha']]

print(
    df_compare[
        ['Class_ID', 'area_ha_raw', 'area_ha_corrected', 'change_ha']
    ].to_string(index=False)
)

 Class_ID  area_ha_raw  area_ha_corrected      change_ha
        1 2.234073e+06       2.349685e+06  115612.084115
        2 2.054927e+06       2.094321e+06   39393.156306
        3 5.448694e+02       7.200787e+00    -537.668648
        4 5.238923e+04       5.613802e+03  -46775.424130
        5 4.108953e+02       9.440067e+02     533.111407
        6 2.842533e+05       1.730719e+05 -111181.470830
        7 0.000000e+00       6.557195e+04   65571.945606
        9 6.512410e+05       5.811897e+05  -70051.339108
       13 0.000000e+00       1.027332e+04   10273.322520
       14 0.000000e+00       6.596897e+03    6596.897024
       16 1.584098e+01       7.751531e+03    7735.690475
       17 4.293445e+03       4.897851e+03     604.406318
       18 1.384838e+05       2.010490e+05   62565.196370
       19 0.000000e+00       1.347435e+04   13474.352813
       20 2.970474e+00       1.408902e+04   14086.045399
       21 0.000000e+00       3.220837e+02     322.083706
       23 2.473436e+05       1.

# Export final LULC map

In [ ]:
# final_corrected_stack = ee.Image.cat([
#     raw_lc,
#     corrected_lc,
#     corrected_mask,
#     no_valid_mask
# ])

# # Export to Earth Engine Asset
# task = ee.batch.Export.image.toDrive(
#     image=final_corrected_stack,
#     description=f'corrected_lc_Aceh_2020_{VERSION}',
#     folder='GEE_exports',
#     region=aoi,
#     scale=100,
#     maxPixels=1e13
# )

# task.start()